[![](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mbanuelos/grad_math_modeling/blob/main/Lectures/Module2_DataSimilarity/06_RecommendationSystems.ipynb#copy=true)


# A Brief Introduction to Recommendation Systems

## Learning objectives

By the end of this lesson, you should be able to:

- Represent ratings as a sparse user–item matrix.
- Contrast content-based, neighborhood, and latent-factor recommendations.
- Produce simple movie neighbors using cosine similarity.
- Identify evaluation and fairness questions for a recommender system.

**Student Learning Outcome (SLO 7):**
> Explain and evaluate a basic recommendation approach using a user–item dataset.


## The recommendation problem

Recommendation systems predict which items a person may value. In a user–item rating matrix, rows can represent movies and columns users; most entries are missing. We will use a small subset of the MovieLens 1M data already included with this module. This is a demonstration, not a production recommender.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity

data_dir = Path("ml-1m")
if not data_dir.exists():
    data_dir = Path("Lectures/Module2_DataSimilarity/ml-1m")

ratings = pd.read_csv(data_dir / "ratings.dat", sep="::", engine="python",
                      names=["user_id", "movie_id", "rating", "timestamp"])
movies = pd.read_csv(data_dir / "movies.dat", sep="::", engine="python", encoding="latin-1",
                     names=["movie_id", "title", "genres"])
ratings.head(), movies.head()


## Part 1 — Item neighborhoods

For speed, select a manageable set of frequently rated movies. We center each user’s observed ratings before calculating cosine similarity, so similarity reflects patterns of above- and below-average ratings rather than only generous or strict raters.


In [ ]:
popular_ids = ratings["movie_id"].value_counts().head(200).index
subset = ratings[ratings["movie_id"].isin(popular_ids)]
item_user = subset.pivot_table(index="movie_id", columns="user_id", values="rating")
centered = item_user.sub(item_user.mean(axis=0), axis=1).fillna(0)
item_similarity = pd.DataFrame(cosine_similarity(centered), index=centered.index, columns=centered.index)

def similar_movies(title_fragment, n=5):
    matches = movies[movies["title"].str.contains(title_fragment, case=False, regex=False)]
    available = matches[matches["movie_id"].isin(item_similarity.index)]
    if available.empty:
        return "No matching movie in the selected subset. Try another title."
    movie_id = available.iloc[0]["movie_id"]
    neighbors = item_similarity.loc[movie_id].drop(movie_id).nlargest(n)
    return movies.set_index("movie_id").loc[neighbors.index, ["title", "genres"]].assign(similarity=neighbors.values)

similar_movies("Toy Story")


### ✏️ Written response 1

Choose a movie in the selected subset and inspect its neighbors. Do the recommendations look plausible? Give one reason a seemingly odd recommendation could still arise from the rating data.

> **YOUR ANSWER:**


## Part 2 — Beyond nearest neighbors

- **Content-based systems** recommend items with similar attributes (such as genres or TF–IDF features).
- **Collaborative filtering** uses patterns across people and items, including neighborhood methods like the one above.
- **Latent-factor methods** use low-rank matrix factorization/SVD-inspired models to estimate hidden preference dimensions.

Offline evaluation must hide known ratings and assess predictions or rankings on that held-out data. Real systems also need to consider popularity bias, cold starts for new users/items, privacy, feedback loops, and whose preferences are well represented.


### Practice and reflection

1. Compare recommendations using raw ratings versus centered ratings. What changes?
2. What does this system do for a new movie with no ratings?
3. Name one way a recommender could narrow rather than expand a user’s choices, and suggest a design response.

> **YOUR ANSWER:**


## Part 0 — Missing is not zero

An empty ratings-matrix cell usually means “not rated,” not “zero stars.” Treating all missing entries as zero distorts similarity. A simple baseline is a popularity list: easy and often strong, but it can overexpose already-popular items.


In [ ]:
movie_summary = (ratings.groupby("movie_id")["rating"].agg(mean_rating="mean", rating_count="size")
                 .join(movies.set_index("movie_id")[["title", "genres"]]))
movie_summary.query("rating_count >= 100").sort_values("mean_rating", ascending=False).head(10)


### Compare two recommenders

Compare cosine-neighbor recommendations for one movie with the popularity list. Which is more personalized? Which may be more dependable for a new user?

> **YOUR ANSWER:**


## Part 3 — Evaluation and holdout data

An offline test hides ratings, builds recommendations from the remaining data, then evaluates hidden items. RMSE measures rating-prediction error; ranking metrics ask whether useful items appear near the top. Never evaluate only on ratings used to build the system.


In [ ]:
eligible = subset["user_id"].value_counts()
eligible = eligible[eligible >= 2].index
held_out = subset[subset["user_id"].isin(eligible)].groupby("user_id", group_keys=False).sample(n=1, random_state=232)
training_rows = subset.drop(held_out.index)
print(f"training ratings: {len(training_rows):,}")
print(f"held-out ratings: {len(held_out):,}")


### Design reflection

What is the cold-start problem for new users and movies? How can popularity create a feedback loop? Propose one control that makes recommendations easier for users to understand or correct.

> **YOUR ANSWER:**


## Looking ahead

Neighborhood methods compare observed ratings directly. Latent-factor methods use low-rank structure to estimate preferences even when few ratings overlap. Neither determines what someone should watch: goals, safeguards, and feedback channels remain design choices.


## Part 4 — Content-based and collaborative signals

Content-based recommendation uses item attributes such as genres, descriptions, or TF–IDF features. Collaborative filtering uses patterns across users and items. Hybrid systems combine them: content can help a new item, while collaborative signals can discover connections not listed in metadata.


### Scenario discussion

A new user rates two science-fiction films highly. Which approach can recommend immediately: content-based, collaborative, or both? What additional information would make the first page of recommendations more useful?

> **YOUR ANSWER:**


## Lesson summary

Recommendation is a prediction-and-ranking problem with sparse, incomplete feedback. Useful systems validate on held-out data and consider diversity, privacy, cold starts, popularity bias, and user control alongside numerical performance.
